In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
%python
source_file=f"{landing_folder_path}/circuits.csv"
table_name=f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Define schema for circuits file with StructType and StructField
circuits_schema = StructType([
    StructField("circuitid", StringType(), True),
    StructField("url", StringType(), True),
    StructField("circuitname", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("locality", StringType(), True),
    StructField("country", StringType(), True)
])

# Read circuits file from volume with defined schema
circuits_df = (spark.read
    .schema(circuits_schema)
    .option("header", True)
    .option("mode", "FAILFAST")
    .csv(source_file)
)

#display(circuits_df)

In [0]:
%python
# Add ingestion timestamp and source file path metadata using helper function
circuits_with_metadata_df = add_ingestion_metadata(circuits_df)

display(circuits_with_metadata_df)

## Understanding `_metadata` Pseudo-Column in Spark

### What is `_metadata`?
`_metadata` is a **hidden pseudo-column** that Spark automatically adds to every DataFrame when reading files. It tracks information about the source file for each row.

### Available Fields in `_metadata`

| Field | Type | Description | Use Case |
|-------|------|-------------|----------|
| **file_path** | string | Full path to source file | Data lineage tracking |
| **file_name** | string | Just the filename | Group by file, identify source |
| **file_size** | long | File size in bytes | Identify large files |
| **file_modification_time** | timestamp | Last modified timestamp | Incremental processing |

### Key Points
* Available in Spark 3.0+
* Works with all file formats (CSV, JSON, Parquet, Delta, etc.)
* **Not visible** in `df.printSchema()` or `df.columns`
* Access using `col("_metadata.field_name")`
* Automatically available - no configuration needed

### Common Use Cases

**1. Data Lineage Tracking**
```python
df.withColumn("source_file", col("_metadata.file_name"))
```

**2. Incremental Processing**
```python
df.filter(col("_metadata.file_modification_time") > "2026-04-20")
```

**3. Multi-File Processing**
```python
df = spark.read.csv("/path/*.csv")
df.groupBy("_metadata.file_name").count()
```

**4. Debugging Data Quality Issues**
```python
df.filter(col("some_column").isNull()) \
  .select("_metadata.file_name", "problematic_column")
```

In [0]:
%python
# Write circuits data to bronze layer table
(circuits_with_metadata_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"✅ Circuits data successfully written to {table_name} table")

In [0]:
SELECT 
  circuitid,
  circuitname,
  locality,
  country,
  lat,
  long,
  ingestion_timestamp,
  source_file_path
FROM formula1.bronze.circuits
LIMIT 10